# Uni-Projekt: Large Language Models from Scratch
**Gruppe:** Phillip, Konstantin, Fabian

**Ziel:** Aufbau und Training eines eigenen LLM-Modells, mithilfe eines Datensatzes der Kochrezepte enthält.

# Datensatz einlesen
* Den CSV Datensatz einlsesn
* Die ersten Zeilen ausgeben

In [1]:
import os
import pandas as pd

file_path = "full_dataset.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    # DataFrame auf die ersten 1000 Zeilen beschränken --> effitienteres entwickeln
    df = df.head(1000)
    print(df.info())
    print(df.head())   
else:    print(f"File '{file_path}' does not exist.")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   title        1000 non-null   object
 2   ingredients  1000 non-null   object
 3   directions   1000 non-null   object
 4   link         1000 non-null   object
 5   source       1000 non-null   object
 6   NER          1000 non-null   object
dtypes: int64(1), object(6)
memory usage: 54.8+ KB
None
   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   
3           3          Chicken Funny   
4           4   Reeses Cups(Candy)     

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small jar chipped beef, cut up", "4 boned ...   
2  ["2 (16 oz.) pkg. frozen corn", "1 (8 oz.) pkg...   
3  ["

- Die CSV-Datei als einen langen Sring formatieren

In [2]:
def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recepie: {title}\nIngredients:{ingredients}\nDirections:{directions}"

raw_text = "".join(df.apply(format_csv, axis=1))
print(f"Der gesamte Datensatz wurde in einen String umgewandelt.")
print(f"Gesamtanzahl der Zeichen: {len(raw_text)}")
print("\nVorschau der ersten 300 Zeichen:")
print(raw_text[:300])

    

Der gesamte Datensatz wurde in einen String umgewandelt.
Gesamtanzahl der Zeichen: 470945

Vorschau der ersten 300 Zeichen:
Recepie: No-Bake Nut Cookies
Ingredients:1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits
Directions:In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and bu


# Tokenizer bauen


- Tokenizer auf mit einfachen Textbeispielen bauen und später auf den Datensatz anwenden
- Mithilfe von RegEx die Wörter trennen

In [3]:
import re

text = "This is a sample text: with numbers 123, special characters !@# and a few more.  ."
def split_text(text):
    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    tokens = [token for token in tokens if token.strip()]
    return tokens


result = split_text(text)


print(result)

['This', 'is', 'a', 'sample', 'text', ':', 'with', 'numbers', '123', ',', 'special', 'characters', '!', '@#', 'and', 'a', 'few', 'more', '.', '.']


- Trennung der Wörter auf den Datensatz anwenden

In [4]:
split = split_text(raw_text)
print(f"Anzahl der Wörter: {len(split)}")
print(split[:20])

Anzahl der Wörter: 110664
['Recepie', ':', 'No-Bake', 'Nut', 'Cookies', 'Ingredients', ':', '1', 'c', '.', 'firmly', 'packed', 'brown', 'sugar', ',', '1/2', 'c', '.', 'evaporated', 'milk']


# Wort-Tokens in Tokens Ids konvertieren

- Vokabular das alle Wort-Tokens im Datensatzenthält

In [5]:
all_words = sorted(set(split))
vocab_size = len(all_words)

print(vocab_size)

3738


- Jedem Token eine ID zuweisen

In [6]:
vocab = {token:integer for integer,token in enumerate(all_words)}

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
('#1', 2)
('#2', 3)
('%', 4)
('&', 5)
("'", 6)
('(', 7)
(')', 8)
('*', 9)
('*Apricot', 10)
('*Can', 11)
('*I', 12)
('+', 13)
('+/-', 14)
('+cooking', 15)
(',', 16)
('-', 17)
('--', 18)
('.', 19)
('0', 20)
('0%', 21)
('1', 22)
('1%', 23)
('1-', 24)
('1-1/2', 25)
('1-8', 26)
('1-gallon', 27)
('1-inch', 28)
('1-quart', 29)
('1/16', 30)
('1/2', 31)
('1/2-inch', 32)
('1/2-pint', 33)
('1/2-quart', 34)
('1/2\\', 35)
('1/2\\tcup', 36)
('1/3', 37)
('1/4', 38)
('1/4-inch', 39)
('1/4\\tturn', 40)
('1/8', 41)
('1/8-inch', 42)
('10', 43)
('10-inch', 44)
('100', 45)
('100%', 46)
('10x', 47)
('11', 48)
('112', 49)
('12', 50)


- Tokenizer Klasse bauen mit der sowohl encoded (Text zu Ids) als auch decoded (Ids zu Texten) werden kann

In [7]:
class TokenizerV1:
    def __init__(self, vocab):
        self.int_to_str = {integer: token for token, integer in vocab.items()}
        self.str_to_int = vocab
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
    

- Jetzt können alle im Vokabular enthaltenen tokens in beliebiger Reihenfolge übergeben werden und in Ids umgewandelt werden
- Ebenfalls kann eine Folge von Ids zurück in Tokens und damit in Text umgewandelt werden

In [8]:
tokenizer = TokenizerV1(vocab)
encoded = tokenizer.encode("baked brown sugar Cookies")
decoded = tokenizer.decode([500, 401, 602, 1503, 95])
print(encoded)
print(decoded)

[1694, 1820, 3436, 457]
Crush Children English Very 270\u00b0


- UNK Token für unbekannte Wörter und end_of_text Token einfügen

In [9]:
all_tokens = sorted(list(set(all_words)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

len(vocab.items())

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('youre', 3735)
('zeppole', 3736)
('zucchini', 3737)
('<|endoftext|>', 3738)
('<|unk|>', 3739)


- Tokenizer anpassen, damit er die neuen Tokens "<|endoftext|>" und "<|unk|>" korrekt verarbeitet

In [10]:
class TokenizerV2:
    def __init__(self, vocab):
        self.int_to_str = {integer: token for token, integer in vocab.items()}
        self.str_to_int = vocab
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]                        
        preprocessed = [
            item if item.strip() in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

- Neuen Tokenizer testen

In [11]:
tokenizer_2 = TokenizerV2(vocab)

text1 = "This is a sample text: with numbers 123, special characters !@# and a few more."
text2 = " And a few more Recepie words and Ingredients Apple, sugar, honey car."

text = " <|endoftext|> ".join([text1, text2])

encoded_v2 = tokenizer_2.encode(text)
print(encoded_v2)

decoded_v2 = tokenizer_2.decode(encoded_v2)
print(decoded_v2)

[1442, 2517, 1598, 3739, 3739, 182, 3708, 3739, 3739, 16, 3351, 3739, 0, 3739, 1650, 1598, 2268, 2744, 19, 3738, 208, 1598, 2268, 2744, 1176, 3739, 1650, 783, 214, 16, 3436, 16, 2480, 3739, 19]
This is a <|unk|> <|unk|> : with <|unk|> <|unk|>, special <|unk|>! <|unk|> and a few more. <|endoftext|> And a few more Recepie <|unk|> and Ingredients Apple, sugar, honey <|unk|>.


## BytePair encoding

- Der einfache Tokeniszer wird durch BytePair encoding ersetzt
- Das erlaubt dem Model Wörter zu unterteilen (sogar bis auf den einzelenen Buchstaben) um neue Wörte die nicht im initial geladenen Vakabular waren in Token Ids zu verwandeln

In [12]:
import tiktoken
import importlib

print ("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


- Tokenizer mit tiktoken erstellen und decoden/encoden

In [15]:
tiktokenizer = tiktoken.get_encoding("gpt2")

text1 = "This is a sample text: with numbers 123, special characters !@# and a few more."
text2 = " And a few more Recepie words and Ingredients Apple, sugar, honey car."

text = " <|endoftext|> ".join([text1, text2])


encoded3 = tiktokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(encoded3) 

decoded3 = tiktokenizer.decode(encoded3)
print(decoded3) 



[1212, 318, 257, 6291, 2420, 25, 351, 3146, 17031, 11, 2041, 3435, 5145, 41573, 290, 257, 1178, 517, 13, 220, 50256, 220, 843, 257, 1178, 517, 19520, 21749, 2456, 290, 33474, 4196, 11, 7543, 11, 12498, 1097, 13]
This is a sample text: with numbers 123, special characters !@# and a few more. <|endoftext|>  And a few more Recepie words and Ingredients Apple, sugar, honey car.
